# Step 4: SQL-Based Business Analysis
Ingests cleaned data into SQL database and performs queries using Joins, Aggregations, CTEs, Window Functions, and Subqueries.


In [1]:
import sqlite3
import os
import sys
import os

def resolve_path(rel_path):
    curr = os.path.abspath(os.getcwd())
    while curr and os.path.dirname(curr) != curr:
        candidate = os.path.join(curr, rel_path)
        if os.path.exists(candidate):
            return os.path.abspath(candidate)
        curr = os.path.dirname(curr)
    return os.path.abspath(rel_path)
import pandas as pd

db_path = resolve_path("data/cleaned/ecommerce.db")
print(f"Connected to database: {db_path}")
conn = sqlite3.connect(db_path)
conn.execute("PRAGMA temp_store = MEMORY;")

query_cte = '''
WITH CustomerSpend AS (
    SELECT 
        c.customer_unique_id,
        COUNT(DISTINCT o.order_id) AS order_count,
        SUM(oi.price + oi.freight_value) AS total_spent
    FROM customers c
    JOIN orders o ON c.customer_id = o.customer_id
    JOIN order_items oi ON o.order_id = oi.order_id
    GROUP BY c.customer_unique_id
)
SELECT 
    customer_unique_id,
    order_count,
    ROUND(total_spent, 2) AS total_spent,
    RANK() OVER (ORDER BY total_spent DESC) AS spending_rank
FROM CustomerSpend
LIMIT 10;
'''

df = pd.read_sql_query(query_cte, conn)
print(df, flush=True)
conn.close()



Connected to database: p:\guvi project 1\data\cleaned\ecommerce.db
                 customer_unique_id  order_count  total_spent  spending_rank
0  0a0a92112bd4c708ca5fde585afaa872            1     13664.08              1
1  da122df9eeddfedc1dc1f5349a1a690c            2      7571.63              2
2  763c8b1c9c68a0229c42c9fc6f662b93            1      7274.88              3
3  dc4802a71eae9be1dd28f5d788ceb526            1      6929.31              4
4  459bef486812aa25204be022145caa62            1      6922.21              5
5  ff4159b92c40ebe40454e3e6a7c35ed6            1      6726.66              6
6  4007669dec559734d6f53e029e360987            1      6081.54              7
7  5d0a2980b292d049061542014e8960bf            1      4809.44              8
8  eebb5dda148d3893cdaf5b5ca3040ccb            1      4764.34              9
9  48e1ac109decbb87765a3eade6854098            1      4681.78             10
